# Phase 3: Bias-Variance Tradeoff

## Step 0: Expectations

**Outcome:** We expect the degree curve to plateau rather than severely overfit. We expect bias to dominate early on because the features (only `temp` is really useful so far) are very limited. The chronological estimate will be worse because we have to extrapolate trend and seasons into the future.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, hashlib
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import GroupKFold

import gradient_descent as core
import polynomial_regression as p2
import bias_variance as p3

PLOTS = Path('plots')
PLOTS.mkdir(exist_ok=True)
P2_STEM = 'polynomial_regression_artifacts'
P3_STEM = 'bias_variance_artifacts'

# Load Phase 2
W_P2, B_P2, META_P2 = p2.load_artifacts(P2_STEM)
print(f'Phase 2 Hash: {META_P2["phase2_w_sha256"]}')



## Step 1: Infrastructure & Data Setup


In [ ]:
train_df = pd.read_csv('Datasets/train.csv')

# Standard Seeded Split (from Phase 1 & 2)
IDS = ['16002932', '16006616', '16004208']
seed = core.compute_seed(IDS)
from sklearn.model_selection import train_test_split
train_set, val_set = train_test_split(train_df, test_size=0.2, random_state=seed)
train_idx = train_set.index.to_numpy()
val_idx = val_set.index.to_numpy()



params = core.fit_preprocessor(train_set)
X_tr_p1, p1_names = core.transform(train_set, params)
X_vl_p1, _ = core.transform(val_set, params)
y_tr = train_set['cnt'].values
y_vl = val_set['cnt'].values

LR = META_P2['lr']
TOL = META_P2['tol']
MAX_ITER = META_P2['max_iter']



## Step 2: Degree sweep (seeded split)


In [ ]:
degrees = range(1, 9)
train_r2s = []
val_r2s = []

for d in degrees:
    spec = {'degrees': {c: d for c in ['temp', 'hum', 'windspeed']}}
    poly_params = p3.fit_poly(X_tr_p1, p1_names, spec)
    X_tr_exp, exp_names = p3.transform_poly(X_tr_p1, p1_names, poly_params)
    X_vl_exp, _ = p3.transform_poly(X_vl_p1, p1_names, poly_params)
    
    w_init = np.zeros(X_tr_exp.shape[1])
    
    w_opt, b_opt, kappa, res = p3.fit_expanded(X_tr_exp, y_tr, X_vl_exp, y_vl, w_init, 0.0, LR, TOL, MAX_ITER)
    
    y_tr_pred = X_tr_exp @ w_opt + b_opt
    y_vl_pred = X_vl_exp @ w_opt + b_opt
    train_r2s.append(r2_score(y_tr, y_tr_pred))
    val_r2s.append(r2_score(y_vl, y_vl_pred))

plt.figure(figsize=(8,4))
plt.plot(degrees, train_r2s, 'b-o', label='Train R2')
plt.plot(degrees, val_r2s, 'r-o', label='Val R2')
plt.legend(); plt.title('Degree Sweep'); plt.show()
plt.savefig(PLOTS / '03_degree_sweep.png')



## Step 3: Learning Curves


In [ ]:
fracs = np.linspace(0.1, 1.0, 10)
lc_train_1 = []; lc_val_1 = []
lc_train_4 = []; lc_val_4 = []

for frac in fracs:
    n_samples = int(frac * len(X_tr_p1))
    idx = np.random.choice(len(X_tr_p1), n_samples, replace=False)
    X_sub = X_tr_p1[idx]; y_sub = y_tr[idx]
    
    # Degree 1
    w_1, b_1, _, _ = p3.fit_expanded(X_sub, y_sub, X_vl_p1, y_vl, np.zeros(X_sub.shape[1]), 0.0, LR, TOL, MAX_ITER)
    lc_train_1.append(r2_score(y_sub, X_sub @ w_1 + b_1))
    lc_val_1.append(r2_score(y_vl, X_vl_p1 @ w_1 + b_1))
    
    # Degree 4
    spec4 = {'degrees': {c: 4 for c in ['temp', 'hum', 'windspeed']}}
    pp4 = p3.fit_poly(X_sub, p1_names, spec4)
    X_s_4, _ = p3.transform_poly(X_sub, p1_names, pp4)
    X_v_4, _ = p3.transform_poly(X_vl_p1, p1_names, pp4)
    w_4, b_4, _, _ = p3.fit_expanded(X_s_4, y_sub, X_v_4, y_vl, np.zeros(X_s_4.shape[1]), 0.0, LR, TOL, MAX_ITER)
    lc_train_4.append(r2_score(y_sub, X_s_4 @ w_4 + b_4))
    lc_val_4.append(r2_score(y_vl, X_v_4 @ w_4 + b_4))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(fracs, lc_train_1, 'b-o', label='Train D=1'); axes[0].plot(fracs, lc_val_1, 'r-o', label='Val D=1'); axes[0].legend()
axes[1].plot(fracs, lc_train_4, 'b-o', label='Train D=4'); axes[1].plot(fracs, lc_val_4, 'r-o', label='Val D=4'); axes[1].legend()
plt.savefig(PLOTS / '03_learning_curves.png')



## Step 6: Test Bias Hypothesis (Interactions)


In [ ]:
spec_inter = {
    'degrees': {'temp': 3, 'hum': 2, 'windspeed': 2},
    'interactions': [('hr', 'workingday'), ('hr', 'weather')]
}
poly_params_int = p3.fit_poly(X_tr_p1, p1_names, spec_inter)
X_tr_int, int_names = p3.transform_poly(X_tr_p1, p1_names, poly_params_int)
X_vl_int, _ = p3.transform_poly(X_vl_p1, p1_names, poly_params_int)

w_init = np.zeros(X_tr_int.shape[1])

w_opt_int, b_opt_int, kappa, res = p3.fit_expanded(X_tr_int, y_tr, X_vl_int, y_vl, w_init, 0.0, LR, TOL, MAX_ITER)

y_tr_pred = X_tr_int @ w_opt_int + b_opt_int
y_vl_pred = X_vl_int @ w_opt_int + b_opt_int
print(f"Interaction Model Train R2: {r2_score(y_tr, y_tr_pred):.4f}")
print(f"Interaction Model Val R2: {r2_score(y_vl, y_vl_pred):.4f}")



## Step 7: Evaluate Under Three Schemes

1. Seeded split (done above)
2. Day-grouped K-Fold
3. Chronological cut


In [ ]:
# 2. Day-grouped K-Fold
gkf = GroupKFold(n_splits=5)
train_df['dteday'] = pd.to_datetime(train_df['dteday'])
groups = train_df['dteday'].dt.date

kf_r2s = []
for tr_idx, vl_idx in gkf.split(train_df, groups=groups):
    tr_set = train_df.iloc[tr_idx]
    vl_set = train_df.iloc[vl_idx]
    pm = core.fit_preprocessor(tr_set)
    X_t, _ = core.transform(tr_set, pm)
    X_v, _ = core.transform(vl_set, pm)
    pp = p3.fit_poly(X_t, p1_names, spec_inter)
    Xt_exp, _ = p3.transform_poly(X_t, p1_names, pp)
    Xv_exp, _ = p3.transform_poly(X_v, p1_names, pp)
    w, b, _, _ = p3.fit_expanded(Xt_exp, tr_set['cnt'].values, Xv_exp, vl_set['cnt'].values, np.zeros(Xt_exp.shape[1]), 0.0, LR, TOL, MAX_ITER)
    kf_r2s.append(r2_score(vl_set['cnt'].values, Xv_exp @ w + b))

print(f"GroupKFold Val R2s: {kf_r2s}\nMean: {np.mean(kf_r2s):.4f}")

# 3. Chronological Cut
chrono_cut = int(len(train_df) * 0.8)
chrono_train = train_df.iloc[:chrono_cut]
chrono_val = train_df.iloc[chrono_cut:]
pm_chr = core.fit_preprocessor(chrono_train)
Xc_tr, _ = core.transform(chrono_train, pm_chr)
Xc_vl, _ = core.transform(chrono_val, pm_chr)
pp_chr = p3.fit_poly(Xc_tr, p1_names, spec_inter)
Xc_tr_exp, _ = p3.transform_poly(Xc_tr, p1_names, pp_chr)
Xc_vl_exp, _ = p3.transform_poly(Xc_vl, p1_names, pp_chr)

wc, bc, _, _ = p3.fit_expanded(Xc_tr_exp, chrono_train['cnt'].values, Xc_vl_exp, chrono_val['cnt'].values, np.zeros(Xc_tr_exp.shape[1]), 0.0, LR, TOL, MAX_ITER)
print(f"Chronological Val R2: {r2_score(chrono_val['cnt'].values, Xc_vl_exp @ wc + bc):.4f}")



## Step 8: Explain the gap with numbers

The drop from Random Seeded Split to Day-Grouped Split is expected because neighbouring hours are not trivially correlated across the split. The drop to Chronological is larger because the model struggles to extrapolate trend into entirely unseen future months.

## Step 10: Save Artifacts for Phase 4


In [ ]:
P3_SHA = p3.save_artifacts(
    P3_STEM,
    w_opt_int, b_opt_int,
    int_names,
    poly_params_int,
    META_P2,
    spec_inter,
    train_idx, val_idx
)
print(f"Saved Phase 3 Artifacts. Hash: {P3_SHA}")

